In [1]:
import duckdb
import glob

Checking if everything works

In [16]:
df = duckdb.query("""
    SELECT *
    FROM read_csv_auto('C:\\Users\\bramm\\studen-grade data\\Data\\raw\\EdNet-KT1\KT1\\*.csv')    
    LIMIT 100
""").to_df()

print(df.head(10))
print(df.columns)

<>:1: SyntaxWarning: invalid escape sequence '\K'
<>:1: SyntaxWarning: invalid escape sequence '\K'
C:\Users\bramm\AppData\Local\Temp\ipykernel_35308\2977817159.py:1: SyntaxWarning: invalid escape sequence '\K'
  df = duckdb.query("""


       timestamp  solving_id question_id user_answer  elapsed_time
0  1565096190868           1       q5012           b         38000
1  1565096221062           2       q4706           c         24000
2  1565096293432           3       q4366           b         68000
3  1565096339668           4       q4829           a         42000
4  1565096401774           5       q6528           b         59000
5  1565096463370           6       q4793           a         58000
6  1565096501746           7       q6488           a         35000
7  1565097101361           8        q356           b         23000
8  1565097171393           9       q1382           c         22000
9  1565097240758          10        q830           b         25000
Index(['timestamp', 'solving_id', 'question_id', 'user_answer',
       'elapsed_time'],
      dtype='str')


Use duckdb to change the dataset into a parquet file for easier reading since the files are too clumsy for reading using python and csv.

In [17]:

con = duckdb.connect()

con.execute("""
    COPY (
        SELECT 
            timestamp::BIGINT,
            solving_id::INT,
            question_id,
            user_answer,
            elapsed_time::INT
        FROM read_csv_auto('C:\\Users\\bramm\\studen-grade data\\Data\\raw\\EdNet-KT1\KT1\\*.csv')
        WHERE solving_id % 10 = 0
    )
    TO 'ednet_subset.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")


<>:3: SyntaxWarning: invalid escape sequence '\K'
<>:3: SyntaxWarning: invalid escape sequence '\K'
C:\Users\bramm\AppData\Local\Temp\ipykernel_35308\803476173.py:3: SyntaxWarning: invalid escape sequence '\K'
  con.execute("""


RuntimeError: Query interrupted

In [2]:
files = glob.glob('C:\\Users\\bramm\\studen-grade data\\Data\\raw\\EdNet-KT1\\KT1\\*.csv')
print(f"Found {len(files)} files.")

# Select the first x amount of files for processing
subset_files = files[:5000]

con = duckdb.connect()

file_list = ','.join([f"'{file}'" for file in subset_files])

con.execute(f"""
    COPY (
        SELECT 
            CAST(timestamp AS BIGINT),
            CAST(solving_id AS INT),
            question_id,
            user_answer,
            CAST(elapsed_time AS INT)
        FROM read_csv([{file_list}])
        WHERE solving_id % 10 = 0
    )
    TO 'ednet_subset.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD);
""")

Found 784309 files.
